# Jacobian lens — GPU 1 demo and fit

**Goal:** inspect a released Jacobian lens, then load a locally trained 100-prompt lens when it is available. Sections 1–7 are the interactive demo; Section 8 verifies the local artifact.

Run from the repository root. The first code cell must run before importing PyTorch; it maps physical GPU 1 to the notebook's `cuda:0`.

## Before you start (clean machine)

You need an NVIDIA GPU with about 24 GB VRAM, a current NVIDIA driver, Python 3.10+, internet for the first model/lens download, and roughly 25 GB of free disk. From the repository root:

```bash
uv venv --python 3.10 --seed .venv
source .venv/bin/activate
pip install 'torch==2.6.0+cu124' --index-url https://download.pytorch.org/whl/cu124
pip install -e . jupyter ipykernel
python -m ipykernel install --sys-prefix --name jlens --display-name 'Python (jlens)'
CUDA_VISIBLE_DEVICES=1 jupyter lab
```

`cu124` is the CUDA PyTorch build validated here; choose a compatible build if your driver differs. Starting Jupyter with `CUDA_VISIBLE_DEVICES=1` is the reliable way to reserve physical GPU 1. Public Hugging Face files are cached after their first download.

In [ ]:
# GPU pinning and cache setup — run this before importing torch.
import os
import sys
from pathlib import Path

if "torch" in sys.modules:
    raise RuntimeError("Restart the kernel, then run this cell first so GPU 1 can be selected.")

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ.setdefault("HF_HOME", "/tmp/hf-cache")
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/tmp/hf-cache/hub")

OUTPUT_ROOT = Path("artifacts/walkthrough-qwen3.5-4b-gpu1")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Physical GPU 1 is mapped to this notebook's cuda:0")

In [ ]:
# Preflight: fail early if packages, CUDA, or GPU selection are wrong.
import importlib.util

required = ["huggingface_hub", "jlens", "torch", "transformers"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
assert not missing, f"Missing dependencies: {missing}. Run .venv/bin/pip install -e . jupyter ipykernel"

import torch

assert torch.cuda.is_available(), "CUDA PyTorch and an available GPU are required."
assert torch.cuda.device_count() == 1, torch.cuda.device_count()
device = torch.device("cuda:0")
print(f"torch={torch.__version__}, CUDA={torch.version.cuda}")
print(f"visible GPU={torch.cuda.get_device_name(0)}")
print(f"free/total bytes={torch.cuda.mem_get_info(0)}")

## 1. Configure the released model and lens

The released lens is compatible only with this exact model architecture. A lens for another model must be fitted separately.

In [ ]:
import jlens

jlens.configure_logging()

MODEL_NAME = "Qwen/Qwen3.5-4B"
LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "qwen-n1000"
LENS_FILE = "qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt"

LOCAL_FITTED_LENS = OUTPUT_ROOT / "jacobian_lens.pt"

print({"model": MODEL_NAME, "local_fitted_lens": str(LOCAL_FITTED_LENS)})

## 2. Load the model on GPU 1

`cuda:0` below is the masked name for physical GPU 1, set in the first cell.

In [ ]:
import transformers

torch.cuda.reset_peak_memory_stats(device)
hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16
).to(device)
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)

assert model.input_device == device
print(model)
print(f"model-load peak GiB={torch.cuda.max_memory_allocated(device) / 2**30:.2f}")

## 3. Load the published Jacobian lens

This is a pre-fitted artifact for the model above. Nothing is trained until Section 8.

In [ ]:
lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename=LENS_FILE, revision=LENS_REVISION
)
lens

## 4. Compare J-lens and vanilla logit lens

Both predict the next token from an intermediate layer. The logit lens decodes it directly; the J-lens estimates the remaining computation first.

In [ ]:
prompt = "Fact: The currency used in the country shaped like a boot is"
layers = [
    model.n_layers // 4,
    model.n_layers // 2,
    model.n_layers // 4 * 3,
    model.n_layers - 2,
]

jlens_logits, model_logits, _ = lens.apply(model, prompt, layers=layers, positions=[-2])
logit_lens, _, _ = lens.apply(
    model, prompt, layers=layers, positions=[-2], use_jacobian=False
)

def top5(logits):
    return [tokenizer.decode([t]) for t in logits.topk(5).indices]

for layer in layers:
    print(f"L{layer:>3} logit-lens: {top5(logit_lens[layer][0])}")
    print(f"L{layer:>3} J-lens:     {top5(jlens_logits[layer][0])}")
print(f"model:           {top5(model_logits[0])}")

## 5. Render an inline interactive slice

The visualisation tracks candidate next tokens across layers. The small D3 fallback only helps restricted notebook environments render the interactive chart.

In [ ]:
import base64
import gzip
import hashlib
import json
import subprocess

from jlens import vis
from jlens.examples import EXAMPLES, resolve_prompt
from jlens.vis import build_page, compute_slice, notebook_iframe

try:
    vis._template("embed")
except RuntimeError:
    d3_body = subprocess.run(
        ["curl", "--fail", "--silent", "--show-error", "-L", vis._D3_URL],
        check=True, capture_output=True,
    ).stdout
    d3_sri = "sha384-" + base64.b64encode(hashlib.sha384(d3_body).digest()).decode()
    assert d3_sri == vis._D3_SRI, f"D3 integrity check failed: {d3_sri}"
    vis._TEMPLATE_FOR_MODE["embed"] = vis.PAGE_TEMPLATE.replace(
        "__D3__", f"<script>\n{d3_body.decode()}\n</script>"
    )

gloss = {int(k): v for k, v in json.load(gzip.open("assets/qwen_gloss.json.gz")).items()}
example = next(e for e in EXAMPLES if e.slug == "multihop")
prompt = resolve_prompt(example, tokenizer)

slice_data = compute_slice(model, lens, prompt, layer_stride=2, mask_display=True)
page, _, _ = build_page(
    slice_data, prompt, title=example.section, description=example.description, alt_token=gloss
)
notebook_iframe(page)

## 6. Render a served slice

Fetch mode writes sidecar files and serves them over a local HTTP server.

In [ ]:
import os
import threading
from functools import partial
from http.server import HTTPServer, SimpleHTTPRequestHandler

example = next(e for e in EXAMPLES if e.slug == "ascii-face")
prompt = resolve_prompt(example, tokenizer)
slice_data = compute_slice(model, lens, prompt, mask_display=True)
out_dir = OUTPUT_ROOT / "slices" / example.slug
out_dir.mkdir(parents=True, exist_ok=True)
page, _, _ = build_page(
    slice_data, prompt, title=example.section, description=example.description,
    alt_token=gloss, mode="fetch", out_dir=out_dir,
)
(out_dir / "index.html").write_text(page)

if "_jlens_httpd" not in globals():
    _handler = partial(SimpleHTTPRequestHandler, directory=os.path.abspath(OUTPUT_ROOT / "slices"))
    _jlens_httpd = HTTPServer(("127.0.0.1", 0), _handler)
    threading.Thread(target=_jlens_httpd.serve_forever, daemon=True).start()
print(f"-> http://localhost:{_jlens_httpd.server_address[1]}/{example.slug}/")

## 7. Available bundled examples

Use these slugs to change the prompt used by either visualisation cell.

In [ ]:
for e in EXAMPLES:
    print(f"{e.slug:>24}  {e.section}")

## 8. Load the local 100-prompt lens

The long fitting job writes `jacobian_lens.pt` after all 100 prompts finish. Its intermediate `ckpt.pt` is only a resumable accumulator, so it cannot be loaded as a lens. This cell skips cleanly until the finished artifact is present.

In [ ]:
if LOCAL_FITTED_LENS.exists():
    fitted_lens = jlens.JacobianLens.load(str(LOCAL_FITTED_LENS))
    assert fitted_lens.d_model == model.d_model
    assert fitted_lens.n_prompts == 100, fitted_lens.n_prompts
    print(f"loaded 100-prompt lens: {LOCAL_FITTED_LENS}")
    fitted_lens
else:
    fitted_lens = None
    print("Local 100-prompt lens is not available yet; Sections 1–7 use the released lens.")

## 9. Completion check

The demo is complete after Section 7. This final check additionally confirms the local 100-prompt artifact when it is available.

In [ ]:
if fitted_lens is not None:
    assert fitted_lens.d_model == model.d_model
    assert fitted_lens.n_prompts == 100
    print("Complete: GPU 1 walkthrough and local 100-prompt lens loaded.")
else:
    print("Complete: GPU 1 walkthrough succeeded; the local 100-prompt lens is still pending.")